# SwiftletCare - train YOLOv8n tren Colab
Notebook train thu tren GPU T4. Chay tung o theo thu tu tren mot runtime moi. Ket noi Google Drive de luu checkpoint.
Val chi co swiftlet: khong dung luot nay de ket luan do chinh xac bon lop. Khong su dung test de chon mo hinh.
ZIP train/val dung ban split_v3 da chuan bi; file trong o E: khong tu xuat hien tren Colab.
Khong bam Run All sau khi da giai nen hoac train: o giai nen tu choi ghi de va o train tao run moi.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or result.stderr)
assert result.returncode == 0, 'GPU check failed; select Colab GPU T4 kernel.'

In [ ]:
import sys
import json
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
environment = {'python': sys.version}
for package in ['torch', 'torchvision', 'ultralytics', 'supervision', 'numpy', 'opencv-python']:
    try:
        environment[package] = version(package)
    except PackageNotFoundError:
        environment[package] = None
print(json.dumps(environment, indent=2))
Path('/content/swiftletcare_environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Kiểm tra ZIP trên Google Drive
Chạy ô bên dưới sau khi đã kết nối Drive. Ô này chỉ kiểm tra file, chưa giải nén hoặc train.

In [ ]:
from pathlib import Path
ZIP = Path('/content/drive/MyDrive/SwiftletCare/swiftletcare_train_val_v3.zip')
if ZIP.is_file():
    print('Da tim thay ZIP!')
    print(f'Dung luong: {ZIP.stat().st_size / 1024**2:.1f} MB')
else:
    print('Chua tim thay ZIP. Can kiem tra ten file va thu muc tren Drive.')

## Extract training data
Verify the archive hash and extract train/val on Colab.

In [ ]:
from pathlib import Path
import hashlib
import zipfile

ZIP = Path('/content/drive/MyDrive/SwiftletCare/swiftletcare_train_val_v3.zip')
ROOT = Path('/content/swiftletcare_dataset')
DATA = ROOT / 'data.yaml'
expected_sha256 = '569cafc1c00d367db764c347a8063a2828f7864ba2e331555bd7f8c59c94a13d'

print('Dang kiem tra ZIP...')
digest = hashlib.sha256()
with ZIP.open('rb') as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(block)
assert digest.hexdigest() == expected_sha256, 'ZIP khong khop ban goc. Can upload lai.'
assert not ROOT.exists(), 'Thu muc dich da ton tai. Dung lai de kiem tra, khong ghi de.'

with zipfile.ZipFile(ZIP) as archive:
    for name in archive.namelist():
        assert (ROOT / name).resolve().is_relative_to(ROOT.resolve()), 'Duong dan ZIP khong hop le'
    assert archive.testzip() is None, 'ZIP bi loi'
    print('ZIP hop le. Dang giai nen...')
    archive.extractall(ROOT)

assert DATA.is_file(), 'Thieu data.yaml'
for split, expected in [('train', 3104), ('val', 936)]:
    images = list((ROOT / split / 'images').iterdir())
    assert len(images) == expected, f'Sai so anh {split}: {len(images)}'
    for image in images:
        assert (ROOT / split / 'labels' / (image.stem + '.txt')).is_file(), f'Thieu nhan: {image.name}'
    print(f'{split}: {len(images)} anh, du nhan')
assert not (ROOT / 'test').exists(), 'Khong dung tap test trong buoc train thu'
print('Giai nen thanh cong!')
print(DATA.read_text(encoding='utf-8'))


## Install training dependencies
Run the next two cells in order. Versions follow Task 13 of the AI Vision plan. No training starts here.

In [ ]:
import subprocess
import sys

# Exact versions from the AI Vision plan; keep Colab's existing torch.
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'ultralytics==8.4.178',
    'supervision==0.30.9',
], check=True)
print('Cai dat xong. Chay o kiem tra phien ban ben duoi.')


In [ ]:
import json
import sys
from pathlib import Path
import torch
import ultralytics
import supervision
from ultralytics import YOLO

assert ultralytics.__version__ == '8.4.178', 'Ultralytics version mismatch; restart kernel and recheck.'
assert supervision.__version__ == '0.30.9', 'Supervision version mismatch; restart kernel and recheck.'
assert torch.cuda.is_available(), 'Torch khong nhan GPU. Dung lai, chua train.'
environment = {
    'python': sys.version,
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
    'supervision': supervision.__version__,
    'gpu': torch.cuda.get_device_name(0),
}
Path('/content/swiftletcare_environment.json').write_text(
    json.dumps(environment, indent=2), encoding='utf-8'
)
print(json.dumps(environment, indent=2))
print('Thu vien va GPU san sang!')


## Train thu YOLOv8n
Run only the next cell. It trains up to 50 epochs (early stopping may finish sooner) and saves checkpoints to Google Drive. Val has only swiftlet, so this run cannot select an official four-class model. Never tune using test.
If CUDA runs out of memory, stop and report the error. Do not use Run All: extraction is deliberately protected from overwriting data.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import torch
import yaml
from ultralytics import YOLO

DATA = Path('/content/swiftletcare_dataset/data.yaml')
DRIVE = Path('/content/drive/MyDrive')
assert DRIVE.is_dir(), 'Chua mount Google Drive. Dung lai de ket noi Drive.'
assert DATA.is_file(), 'Chua giai nen dataset.'
assert torch.cuda.is_available(), 'Chua co GPU.'
config = yaml.safe_load(DATA.read_text(encoding='utf-8'))
assert config['names'] == ['owl', 'rat', 'snake', 'swiftlet'], 'Sai thu tu class ID'

PROJECT = DRIVE / 'SwiftletCare' / 'runs'
RUN_NAME = 'yolov8n_split_v3_trial_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN = PROJECT / RUN_NAME
RUN.mkdir(parents=True, exist_ok=False)
settings = {'epochs': 50, 'imgsz': 640, 'batch': 8, 'seed': 42, 'patience': 15}
(RUN / 'trial-info.json').write_text(json.dumps({
    'dataset': 'split_v3',
    'archiveSha256': '569cafc1c00d367db764c347a8063a2828f7864ba2e331555bd7f8c59c94a13d',
    'limitations': ['Val contains only swiftlet. This is a trial run, not official four-class evaluation.'],
    'settings': settings,
}, indent=2), encoding='utf-8')
shutil.copy2('/content/swiftletcare_environment.json', RUN / 'environment.json')
shutil.copy2(DATA, RUN / 'dataset.yaml')
print('Ket qua se luu tai:', RUN)
print('Train thu; khong su dung tap test.')

model = YOLO('yolov8n.pt')
result = model.train(
    data=str(DATA),
    **settings,
    device=0,
    workers=2,
    project=str(PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    save=True,
    plots=True,
)
print('Train da ket thuc. Thu muc ket qua:', RUN)
print('Trong so:', RUN / 'weights' / 'best.pt')


## View training charts and validation metrics
Run only the next cell after training completes. It reads saved reports without training or evaluating test. Precision, recall and mAP here concern the validation data, which contains only swiftlet.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

# Reuse this session's run; do not start training again.
if 'RUN' in globals():
    REPORT_RUN = Path(RUN)
else:
    candidates = sorted(Path('/content/drive/MyDrive/SwiftletCare/runs').glob('yolov8n_split_v3_trial_*/results.csv'))
    assert len(candidates) == 1, 'Can chon dung thu muc run. Khong chay lai o train.'
    REPORT_RUN = candidates[0].parent

print('Dang xem:', REPORT_RUN)
CSV = REPORT_RUN / 'results.csv'
assert CSV.is_file(), 'Chua thay results.csv. Kiem tra Drive va thu muc run.'
table = pd.read_csv(CSV)
table.columns = table.columns.str.strip()
print('So epoch da ghi:', len(table))
print('Chi so 5 epoch cuoi (khong nhat thiet la epoch cua best.pt):')
columns = [c for c in table.columns if c == 'epoch' or c.startswith('metrics/')]
display(table[columns].tail(5))
print('Val chi co swiftlet; khong dai dien cho ca bon lop.')
for filename in ['results.png', 'BoxPR_curve.png', 'confusion_matrix_normalized.png']:
    figure = REPORT_RUN / filename
    if figure.is_file():
        print(filename)
        display(Image(filename=str(figure), width=1000))
    else:
        print('Khong co file:', filename)
